# 跨市场领先-滞后研究（亚洲半导体 → SOXL）

新采集数据：台湾加权/台积电/韩国SK海力士/KOSPI/日经（亚洲盘，在美股同一本地日开盘前收盘）。
严格区分「同期共振」与「严格滞后预测」，检验亚洲盘是否携带对次日美股 SOXL 的可交易预测信息。


## 1. 预登记计划


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'cross_market_lead_lag_20261010'
plan_bytes = (CARD / 'plan.json').read_bytes()
plan = json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('data source:', plan['new_data_source'])
print('causality:', plan['causality_rule'])


plan sha256: abe0f12ab0ac8c964487af633fbca55e5111e5cb071cc6e9cbdaef2a5a1531eb
data source: data/raw/external/asia_semis_20261010_v1
causality: Only the strictly-lagged alignment (signal at t-1 for the US session at t) counts as forecast evidence. Same-local-date correlation is classified as contemporaneous and is never treated as tradeable.


## 2. 运行（qlib Recorder）


In [2]:
from src.workflow import cross_market_lead_lag as workflow
from src.workflow.cross_market_lead_lag import write_card_files

summary = workflow.run_cross_market_lead_lag(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'])
print('forecast markets:', summary['forecast_markets'])
print('stable excess in development:', summary['stable_excess_in_development'])
print('recorder_id:', summary['recorder_id'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2026/10/09 16:55:40 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[39570:MainThread](2026-10-09 16:55:40,750) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[39570:MainThread](2026-10-09 16:55:40,752) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[39570:MainThread](2026-10-09 16:55:40,752) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[39570:MainThread](2026-10-09 16:55:40,757) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name cross_market_lead_lag_20261010.


[39570:MainThread](2026-10-09 16:55:40,760) INFO - qlib.workflow - [exp.py:258] - Experiment 193308294370187288 starts running ...


[39570:MainThread](2026-10-09 16:55:40,840) INFO - qlib.workflow - [recorder.py:345] - Recorder 1b2cb672588c4bbc9cf276ccfa2a401e starts running under Experiment 193308294370187288 ...


[39570:MainThread](2026-10-09 16:55:40,921) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


decision: no_tradeable_cross_market_forecast
forecast markets: []
stable excess in development: []
recorder_id: 1b2cb672588c4bbc9cf276ccfa2a401e


## 3. 信息系数（同期 vs 严格滞后）


In [3]:
import pandas as pd
ic = pd.read_csv(CARD / 'information_coefficients.csv')
display(ic[['market','region','same_date_ic','lagged_ic','observations','classification']].round(4))


,market,region,same_date_ic,lagged_ic,observations,classification
0,twii,asia,0.1730,0.0031,3278,contemporaneous_only
1,tsmc_tw,asia,0.1484,-0.0123,3285,contemporaneous_only
2,skhynix,asia,0.1809,-0.0004,3271,contemporaneous_only
3,kospi,asia,0.2053,-0.0078,3271,contemporaneous_only
4,nikkei,asia,0.1600,0.0264,3235,contemporaneous_only
5,smh,us_reference,0.9835,-0.0852,3497,us_reference_contemporaneous
6,nvda,us_reference,0.7539,-0.0618,3497,us_reference_contemporaneous
7,tsm,us_reference,0.7505,-0.0624,3497,us_reference_contemporaneous
8,amd,us_reference,0.6451,-0.0405,3497,us_reference_contemporaneous
9,asml,us_reference,0.7901,-0.0573,3497,us_reference_contemporaneous


## 4. 严格滞后策略结果


In [4]:
comp = pd.read_csv(CARD / 'backtest_benchmark_comparison.csv')
dev = comp[(comp.window=='development') & (comp.cost_bps==10.0)]
cols = ['market','wealth_ratio_vs_soxx','sharpe_difference','max_drawdown_improvement_pct','stable_excess']
display(dev[cols].sort_values('sharpe_difference', ascending=False).round(3))
print('development 稳定超额市场:', list(dev.loc[dev.stable_excess, 'market']))


,market,wealth_ratio_vs_soxx,sharpe_difference,max_drawdown_improvement_pct,stable_excess
3,kospi,1.333,-0.068,38.311,False
2,skhynix,0.985,-0.125,39.140,False
0,twii,0.620,-0.212,40.165,False
1,tsmc_tw,0.563,-0.235,38.850,False
4,nikkei,0.332,-0.332,42.439,False
6,nvda,0.196,-0.400,36.178,False
7,tsm,0.116,-0.507,43.099,False
9,asml,0.108,-0.508,42.753,False
8,amd,0.100,-0.520,46.067,False
5,smh,0.053,-0.646,46.840,False


development 稳定超额市场: []


## 5. 中文结论


In [5]:
print('新数据:', summary['new_data_source'])
print('可预测市场:', summary['forecast_markets'] or '无')
print('决定:', summary['decision'], '| promotion_eligible:', summary['promotion_eligible'])
print('qlib Recorder:', summary['recorder_id'])


新数据: data/raw/external/asia_semis_20261010_v1
可预测市场: 无
决定: no_tradeable_cross_market_forecast | promotion_eligible: False
qlib Recorder: 1b2cb672588c4bbc9cf276ccfa2a401e


## 2026-10-14 范围更正

原文将“同日”一概视为不可交易，表述过宽。亚洲当日收盘可以早于美股当日开盘；问题在于本卡的美股收盘到收盘标签包含亚洲信息到达前已发生的收益，而非亚洲/美股现金时段必须同时开放。本卡未检验美股09:47之后的剩余时段。原数值、原计划、Recorder与未晋升决定不变，也不重开lag1切换路线。新的时钟/目标配对检验见[亚洲同日剩余时段研究](../asia_same_session_information_20261014/conclusion.md)，其结果不能由本卡旧相关值代替。
